In [12]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os
import plotly.io as pio
data = "data4"
if data=="data1":
    text = "group"
else:
    text = "random"
# ---------- 配置 ----------
data_folder = os.path.join('Data', data) # 根据你的data变量自动设置

# 新旧算法文件列表（假设curve_names相同）
old_files = ['50_5_0_pre.csv', '50_5_2_pre.csv', '50_5_20_pre.csv', '50_5_Uni_pre.csv']
new_files = ['50_5_0_post.csv', '50_5_2_post.csv', '50_5_20_post.csv', '50_5_Uni_post.csv']
curve_names = ['β = 0', 'β = 2', 'β = 20', 'Uniform']  # 共用名称
colors = px.colors.qualitative.Plotly[:4]  # 统一颜色

# ---------- 数据加载函数 ----------
def load_data(files):
    df_all = pd.DataFrame()
    for file, name in zip(files, curve_names):
        df = pd.read_csv(os.path.join(data_folder, file), index_col='Repetition')
        mean = df.mean(axis=0)
        temp_df = pd.DataFrame({
            'Iter': range(1, 151),
            'Value': mean.values,
            'Condition': name
        })
        df_all = pd.concat([df_all, temp_df])
    return df_all

# 加载新旧算法数据
old_data = load_data(old_files)
new_data = load_data(new_files)

# ---------- 绘图 ----------
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Obs-based", "Global-based"),
    vertical_spacing=0.1
)

# 绘制原始算法（第一子图）
for i, name in enumerate(curve_names):
    df = old_data[old_data['Condition'] == name]
    fig.add_trace(
        go.Scatter(
            x=df['Iter'], y=df['Value'],
            name=name, line=dict(color=colors[i]),
            showlegend=True,
            hovertemplate="Iter: %{x}<br>Value: %{y:.4f}<br>Condition: %{meta[0]}<extra></extra>",
            meta=[name]
        ),
        row=1, col=1
    )

# 绘制新算法（第二子图）
for i, name in enumerate(curve_names):
    df = new_data[new_data['Condition'] == name]
    fig.add_trace(
        go.Scatter(
            x=df['Iter'], y=df['Value'],
            name=name, line=dict(color=colors[i]),
            showlegend=False,  # 避免图例重复
            hovertemplate="Iter: %{x}<br>Value: %{y:.4f}<br>Condition: %{meta[0]}<extra></extra>",
            meta=[name]
        ),
        row=2, col=1
    )

# ---------- 更新布局 ----------
fig.update_layout(
    title_text=f"{text}_50_5",
    title_x=0.5,
    height=1080,                    
    width=1920, 
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.95)
)

# 统一Y轴标签
fig.update_yaxes(title_text="Simple Regret", row=1, col=1)
fig.update_yaxes(title_text="Simple Regret", row=2, col=1)
fig.update_xaxes(title_text="Iteration", row=2, col=1)

fig.show()
pio.write_html(fig, '50_5_simple.html')